# Phase 2 : Expérimentation RAG (Retrieval-Augmented Generation)

Ce notebook démontre l'utilisation de notre système RAG en environnement de développement.
Le RAG permet de pallier les hallucinations des grands modèles de langage en les forçant à se baser sur un contexte documentaire pertinent (nos cours au format PDF).

## Étape 1 : Imports et Configuration
Nous commençons par importer nos modules internes : chargement de documents, base vectorielle FAISS, et notre générateur LLM (Bloomz).

In [7]:
import os
import sys

os.chdir("..")
sys.path.insert(0, ".")

from src.rag_engine.document_loader import load_pdf_documents, get_chunks
from src.rag_engine.vector_store import VectorStoreManager
from src.generator.llm_generator import RAGGenerator
from src.config import *

## Étape 2 : Chargement du Corpus (PDF)
Nous lisons tous les fichiers PDF situés dans notre dossier `data/corpus/` et nous extrayons le texte brut de chaque page.

In [8]:
documents = load_pdf_documents(CORPUS_DIR)
print(f"📄 {len(documents)} pages chargées depuis les PDF")

📄 Reading: '01_Introduction_NLP.pdf'...
  ✅ Extraction successful: 1 pages found.
📄 Reading: '02_Les_Transformers.pdf'...
  ✅ Extraction successful: 1 pages found.
📄 Reading: '03_Systemes_RAG.pdf'...
  ✅ Extraction successful: 1 pages found.

📚 Total: 3 pages extracted from the corpus.
📄 3 pages chargées depuis les PDF


## Étape 3 : Découpage Stratégique (Chunking)
Pour ne pas dépasser la limite de contexte du modèle et pour que la recherche vectorielle soit précise, nous découpons ces pages en petits morceaux (chunks) homogènes, en veillant à ne pas couper les mots en plein milieu.

In [9]:
chunks = get_chunks(documents)
print(f"✂️ {len(chunks)} chunks créés avec succès")

✂️ Chunking completed: 14 chunks created from 3 pages.
✂️ 14 chunks créés avec succès


## Étape 4 : Création de la Base de Données Vectorielle (FAISS)
Chaque chunk de texte est transformé en un vecteur mathématique dense (Embedding) via le modèle `all-MiniLM-L6-v2`. Ces vecteurs sont ensuite stockés dans notre index FAISS pour permettre des recherches sémantiques très rapides.

In [10]:
manager = VectorStoreManager(EMBEDDING_MODEL_NAME, VECTOR_DB_PATH)
manager.build_and_save_index(chunks)

🔄 Loading sentence embedding model: 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5880.85it/s]


✅ Semantic embedding transformer ready!
🔄 Computing semantic vectors for 14 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.73it/s]

📐 Vector dimensional space: 384
✅ FAISS index established with 14 registered vectors.
💾 Binary FAISS vectors successfully written to: 'data/faiss_index'
💾 Accompanying JSON text mappings written to: 'data/faiss_index.docs.json'


## Étape 5 : Test de Recherche Sémantique (Retrieval)
Simulons une question utilisateur. Le système va convertir la question en vecteur, et rechercher dans FAISS les chunks les plus similaires sémantiquement.

In [11]:
query = "Qu'est-ce qu'un système RAG ?"
print(f"🔍 Question posée : {query}\n")

results = manager.search_top_k(query, k=3)
for i, r in enumerate(results, 1):
    print(f"--- Résultat {i} (Source : {r['source']}) ---")
    print(r["text"][:250] + "...\n")

🔍 Question posée : Qu'est-ce qu'un système RAG ?

--- Résultat 1 (Source : 03_Systemes_RAG.pdf - Page 1 (Chunk 1)) ---
Chapitre 3 : RAG (Retrieval-Augmented Generation)
Le RAG, ou Retrieval-Augmented Generation (Génération Augmentée par la Recherche), est un
paradigme novateur conçu pour pallier les défauts inhérents des grands modèles de langage (LLM).
Malgré leur i...

--- Résultat 2 (Source : 01_Introduction_NLP.pdf - Page 1 (Chunk 5)) ---
correctement....

--- Résultat 3 (Source : 03_Systemes_RAG.pdf - Page 1 (Chunk 2)) ---
accéder à des données privées ou
confidentielles d'une entreprise, et surtout, ils sont sujets aux 'hallucinations' : ils peuvent affirmer avec
aplomb des faits totalement faux.
L'architecture RAG résout ces problèmes en introduisant une étape de rec...



## Étape 6 : Génération Augmentée (Generation)
Nous injectons ces documents pertinents dans le 'Prompt' (l'instruction) de notre LLM local (Bloomz). Le modèle générera une réponse basée **uniquement** sur ces informations.

In [12]:
generator = RAGGenerator()
response = generator.generate(query, results)
print("\n🤖 Réponse IA (Avec RAG) :\n", response)

⚠️  Aucun token HuggingFace détecté !
    → Définissez la variable d'environnement HF_TOKEN
    → Ou passez le token en paramètre : RAGGenerator(token='hf_...')
✅ Générateur LLM connecté au modèle 'bigscience/bloomz-560m'


2026-06-19 13:32:14.134 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
Loading weights: 100%|██████████| 293/293 [00:00<00:00, 5178.55it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it st


🤖 Réponse IA (Avec RAG) :
 La recherche augmentée


## Étape 7 : Comparaison et Démonstration de l'Utilité du RAG
Pour prouver la nécessité de notre système, posons la même question au modèle, mais cette fois-ci **sans lui fournir les documents de cours**.

In [13]:
response_no_rag = generator.generate_without_rag(query)
print("❌ Réponse IA (SANS RAG) :\n", response_no_rag)
print("\n✅ Rappel Réponse IA (AVEC RAG) :\n", response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ Réponse IA (SANS RAG) :
 Réactions Agressives de l'organisme

✅ Rappel Réponse IA (AVEC RAG) :
 La recherche augmentée
